# 🏦 Loan Default Prediction using Naïve Bayes

---

## 📌 Problem Statement

A bank wants to **predict whether a loan applicant will default** on their loan (`loan_status = 1`) or not (`loan_status = 0`).  
We will use the **Naïve Bayes algorithm** to build this classification model.

---

## 📂 Dataset Overview

| Column | Description |
|--------|-------------|
| `person_age` | Age of the applicant |
| `person_gender` | Gender |
| `person_education` | Education level |
| `person_income` | Annual income |
| `person_emp_exp` | Years of employment experience |
| `person_home_ownership` | Home ownership type (RENT/OWN/MORTGAGE) |
| `loan_amnt` | Loan amount requested |
| `loan_intent` | Purpose of the loan |
| `loan_int_rate` | Loan interest rate |
| `loan_percent_income` | Loan amount as % of income |
| `cb_person_cred_hist_length` | Credit history length |
| `credit_score` | Credit score |
| `previous_loan_defaults_on_file` | Has defaulted before (Yes/No) |
| `loan_status` | **Target** — 1 = Default, 0 = No Default |

**Total Records:** 45,000

---

## 🧠 What is Naïve Bayes?

Naïve Bayes is a **probabilistic classification algorithm** based on **Bayes' Theorem**:

$$P(Class | Features) = \frac{P(Features | Class) \times P(Class)}{P(Features)}$$

**Why "Naïve"?**  
It assumes that all features are **independent of each other** — which is a simplification (hence "naïve"), but it works surprisingly well in practice!

**Types of Naïve Bayes:**
- `GaussianNB` → for **continuous/numeric** features ✅ (we use this)
- `MultinomialNB` → for **count-based** features (text classification)
- `BernoulliNB` → for **binary** features (0/1)

---

## 🔁 Workflow

```
Load Data → EDA → Preprocessing → Train/Test Split → 
Train Model → Evaluate → Visualize Results
```

## 📦 Step 1: Import Libraries

In [1]:
# Standard Libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Model
from sklearn.naive_bayes import GaussianNB

# Evaluation
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

print("✅ All libraries imported successfully!")

ModuleNotFoundError: No module named 'sklearn'

## 📂 Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('loan_data.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"📋 Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("📌 Dataset Info:")
df.info()

In [ ]:
# Statistical summary
print("📊 Statistical Summary:")
df.describe().round(2)

In [ ]:
# Missing values
print("❓ Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")

In [ ]:
# Target variable distribution
plt.figure(figsize=(6, 4))
target_counts = df['loan_status'].value_counts()
labels = ['No Default (0)', 'Default (1)']
colors = ['#2ecc71', '#e74c3c']

plt.bar(labels, target_counts.values, color=colors, edgecolor='black', width=0.4)
plt.title('Loan Status Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count')
plt.xlabel('Loan Status')
for i, v in enumerate(target_counts.values):
    plt.text(i, v + 200, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📌 Default Rate: {(target_counts[1]/len(df)*100):.1f}%")
print(f"📌 No Default Rate: {(target_counts[0]/len(df)*100):.1f}%")

In [ ]:
# Correlation heatmap (numeric columns only)
plt.figure(figsize=(10, 6))
numeric_cols = df.select_dtypes(include=['float64', 'int64'])
corr = numeric_cols.corr()

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🛠️ Step 4: Data Preprocessing

> **Why preprocessing?**  
> Naïve Bayes (GaussianNB) works with **numeric values only**.  
> We need to convert categorical columns (text) into numbers using **Label Encoding**.

In [ ]:
# Make a copy so original data is safe
df_model = df.copy()

# Identify categorical columns
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f"🔤 Categorical Columns: {cat_cols}")

# Label Encoding
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])
    print(f"  ✅ Encoded: {col}")

print("\n📋 After Encoding — First 3 Rows:")
df_model.head(3)

In [ ]:
# Define Features (X) and Target (y)
X = df_model.drop('loan_status', axis=1)
y = df_model['loan_status']

print(f"🎯 Features (X) shape: {X.shape}")
print(f"🏷️  Target (y) shape:  {y.shape}")
print(f"\n📌 Feature columns: {list(X.columns)}")

## ✂️ Step 5: Train-Test Split

> We split the data into **80% training** and **20% testing**.  
> The model **learns from training data** and is **evaluated on test data** it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📦 Training set size:  {X_train.shape[0]} rows")
print(f"🧪 Testing set size:   {X_test.shape[0]} rows")
print(f"\n✅ stratify=y ensures same class ratio in both sets")

## 🤖 Step 6: Train the Naïve Bayes Model

> **GaussianNB** assumes features follow a **Normal (Gaussian) distribution**.  
> It calculates the **mean** and **variance** of each feature for each class.

In [ ]:
# Initialize and Train the model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

print("✅ Naïve Bayes model trained successfully!")
print(f"\n📌 Classes: {nb_model.classes_}")
print(f"📌 Prior Probabilities: {nb_model.class_prior_.round(3)}")

## 📊 Step 7: Model Evaluation

| Metric | Meaning |
|--------|----------|
| **Accuracy** | % of all correct predictions |
| **Precision** | Of predicted defaults, how many were actually defaults? |
| **Recall** | Of actual defaults, how many did we catch? |
| **F1-Score** | Balance between Precision and Recall |
| **ROC-AUC** | Overall ability to distinguish between classes |

In [ ]:
# Predictions
y_pred = nb_model.predict(X_test)
y_pred_proba = nb_model.predict_proba(X_test)[:, 1]

# Scores
acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_pred_proba)

print(f"🎯 Accuracy  : {acc*100:.2f}%")
print(f"📈 ROC-AUC   : {roc:.4f}")
print()
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontweight='bold')
plt.xlabel('Predicted', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\n✅ True Negatives  (Correctly predicted No Default): {tn}")
print(f"✅ True Positives  (Correctly predicted Default):    {tp}")
print(f"❌ False Positives (Predicted Default, was No Default): {fp}")
print(f"❌ False Negatives (Predicted No Default, was Default): {fn}")

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {roc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
         label='Random Classifier (AUC = 0.50)')
plt.fill_between(fpr, tpr, alpha=0.1, color='orange')
plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('ROC Curve — Naïve Bayes', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔮 Step 8: Predict for a New Loan Applicant

> Let's simulate a **real-world prediction** for a new applicant who has **not been seen by the model** before.

In [ ]:
# New applicant details
# Note: Categorical values were label-encoded. Mapping approximate:
# person_gender:  female=0, male=1
# person_education: Bachelor=0, High School=1, Master=2, PhD=3
# person_home_ownership: MORTGAGE=0, OTHER=1, OWN=2, RENT=3
# loan_intent: DEBTCONSOLIDATION=0, EDUCATION=1, HOMEIMPROVEMENT=2,
#              MEDICAL=3, PERSONAL=4, VENTURE=5
# previous_loan_defaults_on_file: No=0, Yes=1

new_applicant = pd.DataFrame([{
    'person_age': 30,
    'person_gender': 1,            # male
    'person_education': 0,         # Bachelor
    'person_income': 55000,
    'person_emp_exp': 5,
    'person_home_ownership': 3,    # RENT
    'loan_amnt': 10000,
    'loan_intent': 1,              # EDUCATION
    'loan_int_rate': 12.5,
    'loan_percent_income': 0.18,
    'cb_person_cred_hist_length': 4,
    'credit_score': 680,
    'previous_loan_defaults_on_file': 0  # No
}])

prediction = nb_model.predict(new_applicant)[0]
probability = nb_model.predict_proba(new_applicant)[0]

print("🔮 New Applicant Prediction")
print("=" * 40)
print(f"Prediction     : {'⚠️ LOAN DEFAULT' if prediction == 1 else '✅ NO DEFAULT'}")
print(f"P(No Default)  : {probability[0]*100:.1f}%")
print(f"P(Default)     : {probability[1]*100:.1f}%")

## ✅ Step 9: Summary & Key Takeaways

---

### 📌 What We Did

| Step | Action |
|------|--------|
| 1 | Imported libraries |
| 2 | Loaded `loan_data.csv` (45,000 rows, 14 columns) |
| 3 | Performed EDA — checked shape, missing values, distributions |
| 4 | Encoded categorical columns using `LabelEncoder` |
| 5 | Split data into 80% Train / 20% Test |
| 6 | Trained `GaussianNB` model |
| 7 | Evaluated using Accuracy, F1, Confusion Matrix, ROC-AUC |
| 8 | Predicted for a new applicant |

---

### 🧠 Naïve Bayes — Pros and Cons

| ✅ Pros | ❌ Cons |
|---------|--------|
| Very fast to train | Assumes feature independence (rarely true) |
| Works well with small data | Sensitive to feature scaling sometimes |
| Handles multi-class well | Can be outperformed by complex models |
| Good probabilistic output | Continuous features assumed Gaussian |

---

### 🔜 What's Next?

- Try **other classifiers** — Logistic Regression, Random Forest, XGBoost
- Handle **class imbalance** using SMOTE or `class_weight`
- Apply **feature selection** to remove less informative columns
- Use **cross-validation** for more robust evaluation

---

> 💡 **Remember:** No single algorithm wins every time. Always compare multiple models before deploying!